# 📊 BƯỚC 4: ĐÁNH GIÁ MÔ HÌNH AI
**Phụ trách:** Thái (ML Engineer)  
**Yêu cầu:** Chạy `03_training.ipynb` trước để có các biến trong memory, hoặc load model từ file .pkl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Import xong!')

## 1. Load model & metadata từ file

In [ ]:
with open('../models/best_model.pkl', 'rb') as f:
    best_model = pickle.load(f)

with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('../models/label_encoders.pkl', 'rb') as f:
    label_encoders = pickle.load(f)

with open('../models/model_meta.pkl', 'rb') as f:
    meta = pickle.load(f)

print(f'✅ Load model thành công!')
print(f'🏆 Model tốt nhất: {meta["best_model_name"]}')
print(f'📐 Features: {meta["features"]}')

## 2. Bảng kết quả tổng hợp

In [ ]:
all_r = meta['all_results']
df_res = pd.DataFrame(all_r).T[['RMSE','MAE','R2']].rename(columns={'R2':'R² Score'})
df_res = df_res.round(4)
df_res['Xếp hạng R²'] = df_res['R² Score'].rank(ascending=False).astype(int)

print('='*65)
print('📊 BẢNG SO SÁNH KẾT QUẢ 4 MÔ HÌNH ML')
print('='*65)
print(df_res.to_string())
print()
best = meta['best_model_name']
m = meta['metrics']
print(f'🏆 MODEL TỐT NHẤT: {best}')
print(f'   RMSE  = {m["RMSE"]:.4f} tỷ  (~{m["RMSE"]*1000:.0f} triệu VNĐ sai số trung bình)')
print(f'   MAE   = {m["MAE"]:.4f} tỷ')
print(f'   R²    = {m["R2"]:.4f} — Giải thích được {m["R2"]*100:.1f}% biến động giá')

## 3. Rebuild test set để đánh giá chi tiết

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

URL_CC = 'https://raw.githubusercontent.com/tangoctai2004/House-Price-Prediction/refs/heads/main/data/processed/cleaned_chung_cu.csv'
URL_ND = 'https://raw.githubusercontent.com/tangoctai2004/House-Price-Prediction/refs/heads/main/data/processed/cleaned_nha_dat.csv'

df_cc = pd.read_csv(URL_CC)
df_nd = pd.read_csv(URL_ND)
df_cc['loai_bds'] = 'chung_cu'
df_nd['loai_bds'] = 'nha_dat'
if 'balcony_direction' in df_cc.columns:
    df_cc = df_cc.drop(columns=['balcony_direction'])
for c in ['floors_num','frontage_m','road_width_m']:
    df_cc[c] = 0

ALL_COLS = ['price_billion','area_m2','bedrooms_num','district','direction',
            'furniture_std','legal_std','floors_num','frontage_m','road_width_m','loai_bds']
df_all = pd.concat([df_cc[ALL_COLS], df_nd[ALL_COLS]], ignore_index=True)
df_all = df_all.dropna(subset=['price_billion','area_m2'])
df_all = df_all[(df_all['price_billion']>=1) & (df_all['price_billion']<=200)]
df_all = df_all[(df_all['area_m2']>=10) & (df_all['area_m2']<=1000)]

cat_cols = ['district','direction','furniture_std','legal_std','loai_bds']
df_m = df_all.copy()
for col in cat_cols:
    le = label_encoders[col]
    df_m[col] = df_m[col].astype(str).apply(lambda x: le.transform([x])[0] if x in le.classes_ else 0)

FEATURES = meta['features']
X = df_m[FEATURES]
y = df_m['price_billion']
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_pred = best_model.predict(X_test)
print(f'✅ Rebuild test set: {len(y_test)} mẫu')

## 4. Biểu đồ: Giá thực vs Giá dự đoán

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Thực vs Dự đoán
ax = axes[0]
ax.scatter(y_test, y_pred, alpha=0.4, color='#3498db', edgecolors='white', s=30)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Lý tưởng (y=x)')
ax.set_xlabel('Giá thực tế (tỷ VNĐ)')
ax.set_ylabel('Giá dự đoán (tỷ VNĐ)')
ax.set_title(f'Thực tế vs Dự đoán — {meta["best_model_name"]}', fontweight='bold')
ax.legend()
ax.text(0.05, 0.92, f'R² = {r2_score(y_test, y_pred):.4f}', transform=ax.transAxes,
        fontsize=12, color='darkgreen', fontweight='bold')

# Residual plot
ax = axes[1]
residuals = y_test.values - y_pred
ax.scatter(y_pred, residuals, alpha=0.4, color='#e74c3c', edgecolors='white', s=30)
ax.axhline(0, color='black', lw=1.5, ls='--')
ax.set_xlabel('Giá dự đoán (tỷ VNĐ)')
ax.set_ylabel('Residual (Thực - Dự đoán)')
ax.set_title('Phân tích Residual', fontweight='bold')

plt.tight_layout()
plt.savefig('../models/prediction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Lưu biểu đồ phân tích dự đoán!')

## 5. Phân phối sai số dự đoán

In [ ]:
errors = np.abs(y_test.values - y_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram sai số
ax = axes[0]
ax.hist(errors, bins=40, color='#9b59b6', edgecolor='white', alpha=0.85)
ax.axvline(errors.mean(), color='red', lw=2, ls='--', label=f'MAE = {errors.mean():.2f} tỷ')
ax.axvline(np.median(errors), color='orange', lw=2, ls='--', label=f'Median = {np.median(errors):.2f} tỷ')
ax.set_xlabel('Sai số tuyệt đối (tỷ VNĐ)')
ax.set_ylabel('Số lượng')
ax.set_title('Phân phối sai số dự đoán', fontweight='bold')
ax.legend()

# Tỷ lệ sai số trong ngưỡng
ax = axes[1]
thresholds = [1, 2, 3, 5, 10]
pcts = [(errors <= t).mean() * 100 for t in thresholds]
bars = ax.bar([f'≤{t}tỷ' for t in thresholds], pcts, color=['#2ecc71','#27ae60','#f39c12','#e67e22','#e74c3c'])
ax.set_ylabel('% mẫu test')
ax.set_title('Tỷ lệ dự đoán trong ngưỡng sai số', fontweight='bold')
ax.set_ylim(0, 110)
for bar, pct in zip(bars, pcts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{pct:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../models/error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 THỐNG KÊ SAI SỐ:')
print(f'  Sai số trung bình (MAE) : {errors.mean():.3f} tỷ = {errors.mean()*1000:.0f} triệu VNĐ')
print(f'  Sai số trung vị         : {np.median(errors):.3f} tỷ')
print(f'  Sai số max              : {errors.max():.3f} tỷ')
for t, p in zip(thresholds, pcts):
    print(f'  Dự đoán sai ≤ {t:2d} tỷ   : {p:.1f}%')

## 6. Phân tích theo loại BĐS & Quận

In [ ]:
# Thêm cột loai_bds vào X_test để phân tích
X_test_analysis = X_test.copy()
X_test_analysis['y_true'] = y_test.values
X_test_analysis['y_pred'] = y_pred
X_test_analysis['error']  = np.abs(y_test.values - y_pred)

# Decode loai_bds
le_loai = label_encoders['loai_bds']
X_test_analysis['loai_bds_name'] = le_loai.inverse_transform(X_test_analysis['loai_bds'].astype(int))

group = X_test_analysis.groupby('loai_bds_name').agg(
    so_mau=('y_true','count'),
    MAE_tb=('error','mean'),
    R2=('y_true', lambda x: r2_score(x, X_test_analysis.loc[x.index,'y_pred']))
).round(3)

print('📊 Kết quả theo loại BĐS:')
print(group.to_string())

## 7. Kết luận & Gợi ý cho báo cáo

In [ ]:
m = meta['metrics']
print('='*70)
print('📝 KẾT LUẬN CHƯƠNG 4 (DÀNH CHO BÁO CÁO - Mai Anh tổng hợp)')
print('='*70)
print(f'''
Sau khi so sánh 4 thuật toán Machine Learning trên tập dữ liệu bất động
sản Hà Nội gồm chung cư và nhà đất, mô hình **{meta["best_model_name"]}**
cho kết quả tốt nhất:

  • RMSE  = {m["RMSE"]:.4f} tỷ  → sai số bình phương trung bình
  • MAE   = {m["MAE"]:.4f} tỷ  → sai số tuyệt đối trung bình
  • R²    = {m["R2"]:.4f}     → mô hình giải thích {m["R2"]*100:.1f}% biến động giá

So sánh hiệu quả:
  Linear Regression: R² = {all_r["Linear Regression"]["R2"]:.4f} — baseline đơn giản
  Decision Tree    : R² = {all_r["Decision Tree"]["R2"]:.4f} — cải thiện
  Random Forest    : R² = {all_r["Random Forest"]["R2"]:.4f} — khá tốt
  XGBoost          : R² = {all_r["XGBoost"]["R2"]:.4f} — tốt nhất

Các đặc trưng quan trọng nhất: diện tích, quận/huyện, số tầng, mặt tiền.

Hạn chế: dataset còn nhỏ (~1500 bản ghi), một số quận chưa đủ dữ liệu.
Hướng phát triển: thu thập thêm data, thêm tọa độ GPS, fine-tune hyperparameter.
''')
print('='*70)